In [ ]:
# Uncomment and run only if needed:
# %pip install nltk scikit-learn numpy matplotlib spacy gensim transformers torch
# For spaCy's English model, also run in a terminal:
# python -m spacy download en_core_web_sm

In [ ]:
import re, math, random
from collections import Counter, defaultdict
random.seed(42)

# NLTK data downloads (internet required only if resources are not already installed)
import nltk
for resource in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
                 "wordnet", "omw-1.4", "stopwords", "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

def simple_tokens(text):
    return re.findall(r"\b\w+\b|[^\w\s]", text.lower())

print(simple_tokens("NLP is fun! Let's learn."))

In [ ]:
### 1. Text preprocessing: lowercase, clean, tokenize, stopwords, stemming, lemmatization
text = "The Students are studying NLP, and they were running quickly!"
clean = re.sub(r"[^A-Za-z\s]", "", text).lower()
tokens = clean.split()
print("Original:", text)
print("Clean:", clean)
print("Tokens:", tokens)

from nltk.corpus import stopwords
stop = set(stopwords.words("english"))
filtered = [w for w in tokens if w not in stop]
print("Without stopwords:", filtered)

from nltk.stem import PorterStemmer, WordNetLemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
print("Stemming:", [stemmer.stem(w) for w in ["studying", "studies", "running", "better"]])
print("Lemmatization:", [lemmatizer.lemmatize(w, pos="v") for w in ["studying", "studies", "running"]])

In [ ]:
### 2. Unigram, bigram and trigram language model
sentences = ["i love nlp", "i love language", "i enjoy nlp"]
corpus = [s.lower().split() for s in sentences]
unigrams = Counter()
bigrams = Counter()
trigrams = Counter()
for words in corpus:
    words = ["<s>"] + words + ["</s>"]
    unigrams.update(words)
    bigrams.update(zip(words, words[1:]))
    trigrams.update(zip(words, words[1:], words[2:]))
print("Unigrams:", unigrams)
print("Bigrams:", bigrams)
print("Trigrams:", trigrams)
print("Count(i love):", bigrams[("i","love")])

In [ ]:
### 3. Add-one (Laplace) smoothing and sentence probability
train = [s.split() for s in ["i love nlp", "i love language", "i enjoy nlp"]]
uni = Counter(w for sent in train for w in sent)
bi = Counter((sent[i-1], sent[i]) for sent in train for i in range(1, len(sent)))
vocab = set(uni)
V = len(vocab)
def bigram_prob(w_prev, w, alpha=1):
    # P(w | previous) with add-alpha smoothing
    return (bi[(w_prev,w)] + alpha) / (uni[w_prev] + alpha * V)
print("P(love | i):", round(bigram_prob("i","love"), 4))
print("P(language | love):", round(bigram_prob("love","language"), 4))
def sentence_probability(sentence):
    words = sentence.lower().split()
    return math.prod(bigram_prob(words[i-1], words[i]) for i in range(1,len(words)))
print("P('i love nlp'):", sentence_probability("i love nlp"))

In [ ]:
### 4. Perplexity of a bigram model
def perplexity(sentence):
    words = sentence.lower().split()
    probs = [bigram_prob(words[i-1], words[i]) for i in range(1,len(words))]
    if not probs: return 1.0
    return math.exp(-sum(math.log(p) for p in probs) / len(probs))
print("Perplexity:", round(perplexity("i love nlp"), 3))
# Lower perplexity means the model predicts this text better.

In [ ]:
### 5. Byte Pair Encoding (BPE) — tiny educational implementation
def get_pair_counts(words):
    pairs = Counter()
    for word, freq in words.items():
        symbols = word.split()
        for pair in zip(symbols, symbols[1:]):
            pairs[pair] += freq
    return pairs

word_freq = {"l o w": 5, "l o w e r": 2, "n e w": 6, "w i d e r": 3}
for step in range(3):
    pairs = get_pair_counts(word_freq)
    if not pairs: break
    best = pairs.most_common(1)[0][0]
    merged = " ".join(best)
    new_freq = {}
    for word, freq in word_freq.items():
        new_word = word.replace(merged, "".join(best))
        new_freq[new_word] = new_freq.get(new_word, 0) + freq
    word_freq = new_freq
    print("Merge", step+1, best, "=>", word_freq)
print("BPE idea: repeatedly merge the most frequent adjacent symbol pair.")

In [ ]:
### 6. Neural language model (small PyTorch next-word model)
# Optional: requires torch. Tiny demonstration; not a production language model.
try:
    import torch
    import torch.nn as nn
    torch.manual_seed(1)
    sentences = ["i like nlp", "i like python", "you like python", "you love nlp"]
    vocab = sorted(set(" ".join(sentences).split()))
    stoi = {w:i for i,w in enumerate(vocab)}
    pairs = []
    for s in sentences:
        words = s.split()
        pairs += [(stoi[words[i]], stoi[words[i+1]]) for i in range(len(words)-1)]
    X = torch.tensor([p[0] for p in pairs])
    Y = torch.tensor([p[1] for p in pairs])
    model = nn.Sequential(nn.Embedding(len(vocab), 8), nn.Flatten(), nn.Linear(8, len(vocab)))
    opt = torch.optim.Adam(model.parameters(), lr=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(150):
        opt.zero_grad(); loss = loss_fn(model(X), Y); loss.backward(); opt.step()
    with torch.no_grad():
        word = "i"; pred = model(torch.tensor([stoi[word]])).argmax(1).item()
    print("Next word after", word, "->", vocab[pred])
except ImportError:
    print("Install PyTorch with: %pip install torch")

In [2]:
### 7. POS tagging
import nltk
from nltk import word_tokenize, pos_tag
sentence = "The clever student reads books quickly."
try:
    print(pos_tag(word_tokenize(sentence)))
except LookupError:
    print("NLTK tagger data missing; run nltk.download('averaged_perceptron_tagger_eng')")

[('The', 'DT'), ('clever', 'JJ'), ('student', 'NN'), ('reads', 'NNS'), ('books', 'NNS'), ('quickly', 'RB'), ('.', '.')]


In [ ]:
### 8. Context-Free Grammar (CFG) and constituency parsing
from nltk import CFG
from nltk.parse import ChartParser
grammar = CFG.fromstring('''
S -> NP VP
NP -> Det N | 'John'
VP -> V NP | V
Det -> 'the' | 'a'
N -> 'cat' | 'fish'
V -> 'eats' | 'sleeps'
''')
parser = ChartParser(grammar)
words = "the cat eats fish".split()
for tree in parser.parse(words):
    print(tree)
    tree.pretty_print()

In [ ]:
### 9. PCFG (probabilistic CFG)
from nltk import PCFG
from nltk.parse import ViterbiParser
pcfg = PCFG.fromstring('''
S -> NP VP [1.0]
NP -> Det N [0.6] | N [0.4]
VP -> V NP [0.7] | V [0.3]
Det -> 'the' [1.0]
N -> 'cat' [0.5] | 'fish' [0.5]
V -> 'eats' [1.0]
''')
for tree in ViterbiParser(pcfg).parse("the cat eats fish".split()):
    print(tree, "probability =", tree.prob())

In [1]:

# 10. Dependency parsing with spaCy
sentence = "The student reads a book in the library."
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(sentence)
    for token in doc:
        print(f"{token.text:12} POS={token.pos_:6} DEP={token.dep_:10} HEAD={token.head.text}")
except (ImportError, OSError):
    print("Install spaCy/model: %pip install spacy  then  python -m spacy download en_core_web_sm")

The          POS=DET    DEP=det        HEAD=student
student      POS=NOUN   DEP=nsubj      HEAD=reads
reads        POS=VERB   DEP=ROOT       HEAD=reads
a            POS=DET    DEP=det        HEAD=book
book         POS=NOUN   DEP=dobj       HEAD=reads
in           POS=ADP    DEP=prep       HEAD=book
the          POS=DET    DEP=det        HEAD=library
library      POS=NOUN   DEP=pobj       HEAD=in
.            POS=PUNCT  DEP=punct      HEAD=reads


In [ ]:
# 11. WordNet synsets, synonyms, hypernyms and Wu–Palmer similarity
from nltk.corpus import wordnet as wn
word = "car"
print("Synsets:", wn.synsets(word)[:3])
if wn.synsets(word):
    syn = wn.synsets(word)[0]
    print("Definition:", syn.definition())
    print("Lemmas:", [x.name() for x in syn.lemmas()])
    print("Hypernyms:", [x.name() for x in syn.hypernyms()])
a, b = wn.synsets("dog")[0], wn.synsets("cat")[0]
print("Wu-Palmer dog/cat:", a.wup_similarity(b))

In [ ]:
### 12. TF-IDF vector space model and cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
docs = ["cats eat fish", "dogs eat meat", "cats and dogs are pets"]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)
print("Vocabulary:", vectorizer.get_feature_names_out())
print("TF-IDF matrix shape:", X.shape)
print("Cosine similarity:\n", cosine_similarity(X).round(3))

In [ ]:
### 13. Similarity measures: Jaccard and cosine
def jaccard(a, b):
    A, B = set(a.lower().split()), set(b.lower().split())
    return len(A & B) / len(A | B) if A | B else 0
s1, s2 = "nlp is very useful", "nlp is useful"
print("Jaccard:", round(jaccard(s1,s2), 3))
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
mat = CountVectorizer().fit_transform([s1,s2])
print("Cosine:", round(cosine_similarity(mat)[0,1], 3))

In [ ]:
### 14. Word embeddings: Word2Vec (optional) and a simple vector lookup
# Optional gensim Word2Vec; if unavailable, the tiny dictionary example still demonstrates vector lookup.
toy_vectors = {"king":[1.0,0.9,0.1], "queen":[0.9,1.0,0.1], "apple":[0.1,0.1,1.0]}
print("Vector for king:", toy_vectors["king"])
try:
    from gensim.models import Word2Vec
    corpus = [["king","queen","royal"], ["king","man"], ["queen","woman"], ["apple","fruit"]]
    w2v = Word2Vec(sentences=corpus, vector_size=20, window=2, min_count=1, workers=1, seed=1, epochs=100)
    print("Embedding shape:", w2v.wv["king"].shape)
    print("Similar to king:", w2v.wv.most_similar("king", topn=3))
except Exception as e:
    print("Word2Vec optional dependency unavailable:", str(e)[:120])

In [ ]:
### 15. Text classification: TF-IDF + Naive Bayes
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
texts = ["I love this movie", "amazing and wonderful", "I hate this movie",
         "boring and terrible", "great acting", "awful story"]
labels = ["positive","positive","negative","negative","positive","negative"]
clf = make_pipeline(TfidfVectorizer(), MultinomialNB())
clf.fit(texts, labels)
print(clf.predict(["wonderful acting", "terrible boring movie"]))

In [ ]:
### 16. Coreference resolution (simple rule-based demonstration)
text = "Ravi bought a laptop. He uses it every day."
# Very simple pronoun-to-nearest-compatible-mention illustration; not a full coreference system.
sentences = re.split(r"(?<=[.!?])\s+", text)
mentions = ["Ravi", "a laptop", "He", "it"]
clusters = {"Ravi": ["Ravi", "He"], "a laptop": ["a laptop", "it"]}
print("Text:", text)
print("Illustrative coreference clusters:", clusters)
print("Real coreference systems use context, agreement, and mention-pair/coreference models.")

In [ ]:
### 17. RNN-based sequence model (PyTorch)
# Minimal RNN classifier: a sequence of token IDs -> class.
try:
    import torch
    import torch.nn as nn
    class SimpleRNNClassifier(nn.Module):
        def __init__(self, vocab_size=20, embed_dim=8, hidden=12, classes=2):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim)
            self.rnn = nn.RNN(embed_dim, hidden, batch_first=True)
            self.fc = nn.Linear(hidden, classes)
        def forward(self, x):
            embedded = self.embedding(x)
            _, hidden = self.rnn(embedded)
            return self.fc(hidden[-1])
    model = SimpleRNNClassifier()
    sample = torch.tensor([[1,2,3,4],[2,3,4,5]])
    print("Logits shape:", model(sample).shape) # [batch_size, number_of_classes]
except ImportError:
    print("Install PyTorch: %pip install torch")

In [ ]:
### 18. Machine translation (toy dictionary baseline)
# Offline, deterministic toy translation. Real MT uses trained neural models.
translation = {"hello":"hola", "good":"bueno", "morning":"mañana", "world":"mundo"}
def translate(sentence):
    return " ".join(translation.get(w.lower(), "[UNK]") for w in sentence.split())
print(translate("hello world"))
print(translate("good morning"))
# For a pretrained translator, use a Hugging Face translation model (internet/model download required).

In [ ]:
### 19. Evaluation metrics: accuracy, precision, recall, F1, BLEU
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
y_true = ["pos","neg","pos","pos","neg"]
y_pred = ["pos","neg","neg","pos","neg"]
print("Accuracy:", accuracy_score(y_true,y_pred))
print("Precision:", precision_score(y_true,y_pred,pos_label="pos"))
print("Recall:", recall_score(y_true,y_pred,pos_label="pos"))
print("F1:", f1_score(y_true,y_pred,pos_label="pos"))
print(classification_report(y_true,y_pred,zero_division=0))
# BLEU for machine translation
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    reference = [["the","cat","is","here"]]
    candidate = ["the","cat","is","there"]
    print("BLEU:", sentence_bleu(reference, candidate, smoothing_function=SmoothingFunction().method1))
except Exception as e:
    print("BLEU demo unavailable:", e)

In [ ]:
### 20. Scaled dot-product attention (NumPy)
import numpy as np
Q = np.array([[1.,0.],[0.,1.]])   # queries
K = np.array([[1.,0.],[0.,1.],[1.,1.]]) # keys
V = np.array([[10.,0.],[0.,10.],[5.,5.]]) # values
scores = Q @ K.T / math.sqrt(K.shape[1])
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights = weights / weights.sum(axis=1, keepdims=True)
output = weights @ V
print("Attention weights:\n", weights.round(3))
print("Attention output:\n", output.round(3))

In [ ]:
### 21. Transformer encoder (PyTorch)
try:
    import torch
    import torch.nn as nn
    layer = nn.TransformerEncoderLayer(d_model=16, nhead=4, batch_first=True)
    encoder = nn.TransformerEncoder(layer, num_layers=2)
    x = torch.rand(2, 5, 16)  # batch=2, sequence length=5, embedding size=16
    y = encoder(x)
    print("Input shape:", x.shape, "Output shape:", y.shape)
except ImportError:
    print("Install PyTorch: %pip install torch")

In [ ]:
### 22. BERT: pretrained sentiment pipeline (download required)
# This downloads a pretrained model the first time; internet is required.
try:
    from transformers import pipeline
    sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
    print(sentiment("The lab test was easier than I expected."))
except Exception as e:
    print("BERT/Transformers model not available offline. Install transformers + torch and run with internet.")
    print("Reason:", str(e)[:250])

In [ ]:
### 23. Named Entity Recognition (NER) using spaCy
text = "Sundar Pichai works at Google in California."
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)
    for ent in doc.ents:
        print(ent.text, "->", ent.label_)
except (ImportError, OSError):
    print("Install spaCy model: python -m spacy download en_core_web_sm")